# Level 3D — Stress Testing Foundation

**Audience:** analysts who need transparent, probability-free portfolio stress
tests after learning simulation and model validation.

**Prerequisites:** Levels 1–3C and basic pandas indexing.

**Learning goals**

1. distinguish deterministic stress scenarios from Monte Carlo probabilities;
2. construct historical and hypothetical labelled asset shocks;
3. attribute portfolio stress returns to assets and test loss limits;
4. evaluate multi-period stress paths, terminal loss, and maximum drawdown.

All data are synthetic and no scenario is investment advice or a forecast.

## 1. Setup

Returns and shocks are decimal simple returns. Portfolio weights are labelled,
fully invested, and may include short positions. Thresholds are loss limits,
not VaR confidence levels.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.stress import (
    historical_stress_scenarios,
    stress_test_portfolio,
    stress_test_portfolio_paths,
)

## 2. Build historical window scenarios

The helper compounds each asset's periodic returns over an inclusive window.
The example dates are synthetic labels; they do not recreate a real crisis.

In [ ]:
dates = pd.date_range("2025-01-31", periods=12, freq="ME")
asset_returns = pd.DataFrame(
    {
        "Global Equity": [
            0.03, -0.04, -0.12, -0.08, 0.05, 0.02,
            0.01, -0.02, 0.04, 0.03, -0.01, 0.02,
        ],
        "Government Bond": [
            0.01, 0.02, 0.03, 0.01, -0.01, 0.00,
            0.01, 0.01, -0.02, 0.00, 0.01, 0.01,
        ],
        "Credit": [
            0.01, -0.01, -0.04, -0.03, 0.02, 0.01,
            0.01, 0.00, 0.02, 0.01, 0.00, 0.01,
        ],
    },
    index=dates,
)

historical = historical_stress_scenarios(
    asset_returns,
    {
        "synthetic_selloff": (dates[1], dates[3]),
        "synthetic_recovery": (dates[4], dates[6]),
    },
)
historical

## 3. Add hypothetical shocks

Hypothetical scenarios are explicit asset-return assumptions. Combining them
with historical windows does not attach probability to either set.

In [ ]:
hypothetical = pd.DataFrame(
    {
        "Global Equity": [-0.25, -0.10, 0.05],
        "Government Bond": [0.04, -0.12, -0.03],
        "Credit": [-0.08, -0.15, -0.05],
    },
    index=[
        "equity_crash",
        "rates_and_credit",
        "inflation_pressure",
    ],
)

scenarios = pd.concat([historical, hypothetical])
scenarios

## 4. Test a labelled portfolio

Asset contributions are weight × scenario return and add to portfolio return.
`portfolio_loss` is the signed negative of return, so a gain appears as a
negative loss.

In [ ]:
weights = pd.Series(
    {
        "Global Equity": 0.50,
        "Government Bond": 0.30,
        "Credit": 0.20,
    },
    name="strategic_weight",
)

stress = stress_test_portfolio(
    weights,
    scenarios,
    loss_thresholds={
        "warning_8pct": 0.08,
        "capital_limit_12pct": 0.12,
    },
)
stress.summary.sort_values("portfolio_loss", ascending=False)

In [ ]:
stress.asset_contributions.loc[
    stress.summary["portfolio_loss"].idxmax()
].sort_values()

In [ ]:
stress.threshold_breaches

## 5. Evaluate multi-period paths

Path stress testing preserves sequencing. The current contract resets to the
supplied weights each period and excludes transaction costs. Terminal loss
limits and maximum drawdown answer different questions.

In [ ]:
scenario_paths = {
    "fast_crash_then_rebound": pd.DataFrame(
        {
            "Global Equity": [-0.20, -0.12, 0.10, 0.06],
            "Government Bond": [0.02, 0.01, -0.01, 0.00],
            "Credit": [-0.06, -0.04, 0.03, 0.02],
        }
    ),
    "persistent_rates_shock": pd.DataFrame(
        {
            "Global Equity": [-0.03, -0.02, 0.00, 0.01],
            "Government Bond": [-0.05, -0.04, -0.03, 0.01],
            "Credit": [-0.04, -0.03, -0.02, 0.01],
        }
    ),
}

path_stress = stress_test_portfolio_paths(
    weights,
    scenario_paths,
    loss_thresholds={"terminal_limit_10pct": 0.10},
)
path_stress.summary

In [ ]:
path_stress.portfolio_returns

## 6. Exercise — concentration sensitivity

Change the portfolio to 70% Global Equity, 20% Government Bond, and 10% Credit.
Compare the worst scenario, portfolio loss, asset contributions, and threshold
breaches. Do not assume the worst asset shock is automatically the largest
portfolio contributor.

In [ ]:
concentrated_weights = pd.Series(
    {
        "Global Equity": 0.70,
        "Government Bond": 0.20,
        "Credit": 0.10,
    }
)

# Run stress_test_portfolio and compare its summary with `stress.summary`.

### Answer scaffold

In [ ]:
concentrated = stress_test_portfolio(
    concentrated_weights,
    scenarios,
    loss_thresholds={
        "warning_8pct": 0.08,
        "capital_limit_12pct": 0.12,
    },
)

pd.concat(
    {
        "strategic": stress.summary["portfolio_loss"],
        "concentrated": concentrated.summary["portfolio_loss"],
    },
    axis=1,
).sort_values("concentrated", ascending=False)

## Interpretation and limitations

- Stress tests answer “what if,” not “how likely.”
- Scenario labels, dates, shocks, weights, limits, and aggregation rules belong
  in the decision record.
- Historical windows depend on the chosen sample and do not bound future loss.
- Hypothetical shocks should have governance ownership and documented rationale.
- Static one-period contribution is additive; multi-period compounded
  attribution needs a separate contract.
- The path API assumes constant-period rebalancing and excludes costs,
  liquidity, taxes, market impact, and forced deleveraging.
- Correlated scenario generation remains a separate future module.

Next: correlated multivariate scenarios can generate richer paths for this
stress layer without turning deterministic stresses into probabilities.